# Practical 3 — Transformer Language Models

**Enrollment:** BT24S05F007

## Aim
Study how a transformer turns tokens into vectors and lets every token look at the others, then generate a continuation with a pretrained language model.

## Pipeline used in this practical
1. Split a sentence into tokens.
2. Replace each token with the same embedding whenever the word repeats.
3. Compute scaled dot-product self-attention and read the weight matrix.
4. Ask a pretrained DistilGPT-2 model to continue a prompt.


In [1]:
import numpy as np

rng = np.random.default_rng(24)


## Tokens and embeddings
The sentence is split on spaces. A small random vector is stored once per distinct word, then reused. The two copies of "cat" therefore enter attention as the same vector, which makes the weight matrix easy to read.


In [2]:
words = ["the", "cat", "sat", "because", "the", "cat", "was", "tired"]
unique_words = list(dict.fromkeys(words))
vector_size = 4
word_vector = {word: rng.normal(0.0, 1.0, size=vector_size) for word in unique_words}
token_matrix = np.vstack([word_vector[word] for word in words])

print("Tokens with indexes:")
for index, word in enumerate(words):
    print(f"  {index}: {word}")
print("Embedding matrix shape (tokens, vector size):", token_matrix.shape)
print("Same vector reused for both 'cat' tokens:", np.allclose(token_matrix[1], token_matrix[5]))


Tokens with indexes:
  0: the
  1: cat
  2: sat
  3: because
  4: the
  5: cat
  6: was
  7: tired
Embedding matrix shape (tokens, vector size): (8, 4)
Same vector reused for both 'cat' tokens: True


## Self-attention
For token matrix `X`, the score matrix is `X Xᵀ / sqrt(d)`. Softmax across each row turns scores into weights that sum to 1. The new vector for a token is the weighted sum of every token's vector.

`weights[i, j]` is how much token `i` uses token `j`.


In [3]:
def self_attention(token_vectors):
    width = token_vectors.shape[1]
    scores = (token_vectors @ token_vectors.T) / np.sqrt(width)
    scores = scores - scores.max(axis=1, keepdims=True)
    weights = np.exp(scores)
    weights = weights / weights.sum(axis=1, keepdims=True)
    mixed = weights @ token_vectors
    return weights, mixed

attention, mixed_tokens = self_attention(token_matrix)
rounded = np.round(attention, 3)

print("Attention weights (rows attend to columns):")
header = " ".join(f"{word:>8}" for word in words)
print(" " * 8 + header)
for word, row in zip(words, rounded):
    cells = " ".join(f"{value:8.3f}" for value in row)
    print(f"{word:>8} {cells}")

print("Each row sums to 1:", np.allclose(attention.sum(axis=1), 1.0))
focus = int(np.argmax(attention[5]))
print(f"Second 'cat' (index 5) puts the most weight on index {focus} ('{words[focus]}').")
print("Context-vector shape after mixing:", mixed_tokens.shape)


Attention weights (rows attend to columns):
             the      cat      sat  because      the      cat      was    tired
     the    0.334    0.038    0.103    0.030    0.334    0.038    0.052    0.070
     cat    0.061    0.296    0.052    0.130    0.061    0.296    0.069    0.035
     sat    0.131    0.042    0.366    0.163    0.131    0.042    0.065    0.062
 because    0.034    0.091    0.143    0.570    0.034    0.091    0.026    0.010
     the    0.334    0.038    0.103    0.030    0.334    0.038    0.052    0.070
     cat    0.061    0.296    0.052    0.130    0.061    0.296    0.069    0.035
     was    0.070    0.059    0.069    0.032    0.070    0.059    0.209    0.433
   tired    0.034    0.011    0.023    0.004    0.034    0.011    0.154    0.731
Each row sums to 1: True
Second 'cat' (index 5) puts the most weight on index 1 ('cat').
Context-vector shape after mixing: (8, 4)


## Two heads
One attention pattern is only one view of the sentence. A second head uses a different linear map of the same embeddings, so it can highlight a different relationship. The two context vectors are concatenated. That is the multi-head step in a transformer block, at a size small enough to print.


In [4]:
def project(vectors, seed):
    local = np.random.default_rng(seed)
    matrix = local.normal(0.0, 1.0 / np.sqrt(vectors.shape[1]), size=(vectors.shape[1], vectors.shape[1]))
    return vectors @ matrix

head_one, _ = self_attention(project(token_matrix, seed=3))
head_two, _ = self_attention(project(token_matrix, seed=11))

print("Head 1, weight from the second 'cat' onto each token:")
print(dict(zip(words, np.round(head_one[5], 3))))
print("Head 2, weight from the second 'cat' onto each token:")
print(dict(zip(words, np.round(head_two[5], 3))))
print("Heads agree on the top token:", words[int(np.argmax(head_one[5]))] == words[int(np.argmax(head_two[5]))])


Head 1, weight from the second 'cat' onto each token:
{'the': np.float64(0.001), 'cat': np.float64(0.326), 'sat': np.float64(0.008), 'because': np.float64(0.312), 'was': np.float64(0.023), 'tired': np.float64(0.004)}
Head 2, weight from the second 'cat' onto each token:
{'the': np.float64(0.078), 'cat': np.float64(0.182), 'sat': np.float64(0.088), 'because': np.float64(0.235), 'was': np.float64(0.093), 'tired': np.float64(0.064)}
Heads agree on the top token: False


## Text generation with a pretrained model
DistilGPT-2 is a small public transformer already trained to predict the next word. The practical does not train it. It only loads the weights and continues one prompt. Sampling is seeded so the paragraph stays the same on a repeat run.


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(24)
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")
model.eval()

prompt = "A generative model creates new examples by"
token_ids = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    generated_ids = model.generate(
        **token_ids,
        max_new_tokens=45,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        pad_token_id=tokenizer.eos_token_id,
    )
paragraph = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(paragraph)


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

A generative model creates new examples by building on a simple model and applying the properties to the object that are actually there.




But the result is that the method has some useful properties, it has no way of generating new lists. Instead,


## Conclusion
Self-attention builds a new vector for each token by mixing the others, and the printed matrix shows that mix directly. Repeating the step with a second projection gives a second head. A pretrained transformer uses many such blocks, plus a next-token classifier, which is why loading DistilGPT-2 is enough to continue a sentence without training the network in this practical.


## Run results

These are the values produced when the notebook was executed.

```
Tokens with indexes:
  0: the
  1: cat
  2: sat
  3: because
  4: the
  5: cat
  6: was
  7: tired
Embedding matrix shape (tokens, vector size): (8, 4)
Same vector reused for both 'cat' tokens: True
```

```
Attention weights (rows attend to columns):
             the      cat      sat  because      the      cat      was    tired
     the    0.334    0.038    0.103    0.030    0.334    0.038    0.052    0.070
     cat    0.061    0.296    0.052    0.130    0.061    0.296    0.069    0.035
     sat    0.131    0.042    0.366    0.163    0.131    0.042    0.065    0.062
 because    0.034    0.091    0.143    0.570    0.034    0.091    0.026    0.010
     the    0.334    0.038    0.103    0.030    0.334    0.038    0.052    0.070
     cat    0.061    0.296    0.052    0.130    0.061    0.296    0.069    0.035
     was    0.070    0.059    0.069    0.032    0.070    0.059    0.209    0.433
   tired    0.034    0.011    0.023    0.004    0.034    0.011    0.154    0.731
Each row sums to 1: True
Second 'cat' (index 5) puts the most weight on index 1 ('cat').
Context-vector shape after mixing: (8, 4)
```

```
Head 1, weight from the second 'cat' onto each token:
{'the': np.float64(0.001), 'cat': np.float64(0.326), 'sat': np.float64(0.008), 'because': np.float64(0.312), 'was': np.float64(0.023), 'tired': np.float64(0.004)}
Head 2, weight from the second 'cat' onto each token:
{'the': np.float64(0.078), 'cat': np.float64(0.182), 'sat': np.float64(0.088), 'because': np.float64(0.235), 'was': np.float64(0.093), 'tired': np.float64(0.064)}
Heads agree on the top token: False
```

```
A generative model creates new examples by building on a simple model and applying the properties to the object that are actually there.




But the result is that the method has some useful properties, it has no way of generating new lists. Instead,
```
